In [7]:
import os
import json
import re
from typing import Dict, Set, Any

import pandas as pd
from tqdm import tqdm

CHOICE_TYPES = {"single_choice", "multiple_choice"}
RATE_TYPES = {"multiple_rate"}  # <-- NEW
MULTI_DELIM = "^"
RATE_DELIM = "$"                            # <-- NEW
CSV_ENCODING = "ISO-8859-1"

# Regex to remove leading/trailing whitespace INCLUDING NBSP (\u00A0) and typical Unicode spaces
TRIM_RE = re.compile(r"^[\s\u00A0\u200B\u200E\u200F]+|[\s\u00A0\u200B\u200E\u200F]+$")


def clean_opt(x: Any) -> str:
    """
    Case-sensitive option cleaning:
    - convert to string
    - normalize NBSP and a few invisible chars
    - remove leading/trailing whitespace robustly (including \xa0)
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x)
    # common "invisible" characters
    s = s.replace("\u00A0", " ")  # NBSP
    s = s.replace("\u200B", "")   # zero-width space
    s = s.replace("\u200E", "")   # LRM
    s = s.replace("\u200F", "")   # RLM
    # robust trim
    s = TRIM_RE.sub("", s)
    return s


def load_question_types(json_path: str) -> Dict[str, str]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out: Dict[str, str] = {}
    for meta in data.values():
        if isinstance(meta, dict):
            new_name = meta.get("new_name")
            qtype = meta.get("question_type")
            if isinstance(new_name, str) and isinstance(qtype, str):
                out[new_name] = qtype
    return out


def load_option_maps(options_csv_path: str) -> Dict[str, Dict[str, str]]:
    """
    {question_new_name: {clean(old_option): clean(new_option)}}
    - Case-sensitive
    - If new_option is empty -> do not map (keep old option)
    """
    m = pd.read_csv(
        options_csv_path,
        dtype=str,
        keep_default_na=False,
        encoding=CSV_ENCODING,
    )

    required = {"new_name", "old_option", "new_option"}
    missing = required - set(m.columns)
    if missing:
        raise ValueError(f"options_csv missing columns: {sorted(missing)}")

    per_question: Dict[str, Dict[str, str]] = {}
    for _, row in m.iterrows():
        q = clean_opt(row.get("new_name"))
        old = clean_opt(row.get("old_option"))
        new = clean_opt(row.get("new_option"))

        if not q or not old:
            continue
        if new == "":
            continue  # keep old option as-is

        per_question.setdefault(q, {})[old] = new

    return per_question


def map_single_choice(value: Any, mapping: Dict[str, str]) -> Any:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value

    key = clean_opt(value)
    if key == "":
        return value

    mapped = mapping.get(key)
    return mapped if mapped is not None else value


def map_multiple_choice(value: Any, mapping: Dict[str, str]) -> Any:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value

    s = str(value)
    if clean_opt(s) == "":
        return value

    parts = s.split(MULTI_DELIM)
    out_parts = []
    for p in parts:
        token = clean_opt(p)
        if token == "":
            continue
        out_parts.append(mapping.get(token, token))  # case-sensitive

    return MULTI_DELIM.join(out_parts)


def map_rate(value: Any, mapping: Dict[str, str]) -> Any:
    """
    Map only the ASPECT part of rate strings.

    Handles:
      - "aspect$rate"
      - "aspect$rate^aspect$rate^..."
      - also tolerates tokens without '$' by mapping whole token (rare/malformed)

    Keeps rate untouched.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value

    s = str(value)
    if clean_opt(s) == "":
        return value

    tokens = s.split(MULTI_DELIM)  # works even if there's no '^'
    out_tokens = []

    for t in tokens:
        t_clean = clean_opt(t)
        if t_clean == "":
            continue

        if RATE_DELIM in t_clean:
            aspect_raw, rate_raw = t_clean.split(RATE_DELIM, 1)
            aspect = clean_opt(aspect_raw)
            rate = clean_opt(rate_raw)

            mapped_aspect = mapping.get(aspect, aspect)  # map aspect only
            # keep the original rate part (cleaned)
            out_tokens.append(f"{mapped_aspect}{RATE_DELIM}{rate}")
        else:
            # If somehow there's no '$', treat it like an aspect token
            out_tokens.append(mapping.get(t_clean, t_clean))

    return MULTI_DELIM.join(out_tokens)


def main():
    current_dir = os.getcwd()

    csv_path = os.path.join(current_dir, r"organized\3_2022_columnNamesMapped.csv")
    json_path = os.path.join(current_dir, r"Helper Data\column_mappings_new.json")
    options_csv = os.path.join(current_dir, r"Helper Data\option_mapping_final.csv")
    output_path = os.path.join(current_dir, r"organized\4_2022_optionsMapped.csv")

    df = pd.read_csv(csv_path, dtype=str, encoding=CSV_ENCODING)
    qtypes = load_question_types(json_path)
    option_maps = load_option_maps(options_csv)

    target_types = CHOICE_TYPES | RATE_TYPES
    target_columns: Set[str] = {c for c in df.columns if qtypes.get(c) in target_types}

    print(f"\nProcessing {len(target_columns)} questions (choice + rate; case-sensitive, robust whitespace trim)...\n")

    for col in tqdm(sorted(target_columns), desc="Questions", unit="question"):
        mapping = option_maps.get(col, {})
        if not mapping:
            continue

        tqdm.pandas(desc=f"Mapping '{col}'")
        qtype = qtypes[col]

        if qtype == "single_choice":
            df[col] = df[col].progress_apply(lambda v: map_single_choice(v, mapping))
        elif qtype == "multiple_choice":
            df[col] = df[col].progress_apply(lambda v: map_multiple_choice(v, mapping))
        elif qtype in RATE_TYPES:
            df[col] = df[col].progress_apply(lambda v: map_rate(v, mapping))

    df.to_csv(output_path, index=False, encoding=CSV_ENCODING)
    print("\nDone ✅")


if __name__ == "__main__":
    main()



Processing 45 questions (choice + rate; case-sensitive, robust whitespace trim)...



Mapping 'accompanying_people': 100%|██████████| 1527/1527 [00:00<00:00, 508996.44it/s]

Mapping 'adequate_bike_spaces_stop': 100%|██████████| 1527/1527 [00:00<00:00, 728966.79it/s]

Mapping 'age_group': 100%|██████████| 1527/1527 [00:00<00:00, 373212.65it/s]

Mapping 'annual_cost_public_subscription_transport': 100%|██████████| 1527/1527 [00:00<00:00, 299453.07it/s]

Mapping 'app_bike_walking_pt_gamification_rewards_willingness_to_use': 100%|██████████| 1527/1527 [00:00<00:00, 504346.97it/s]

Mapping 'are_ebooks_brochures_usefull_for_information_on_sustainable_mobility': 100%|██████████| 1527/1527 [00:00<00:00, 502290.19it/s]

Mapping 'are_you_aware_of_the_different_services_offered_by_the_portal': 100%|██████████| 1527/1527 [00:00<00:00, 757952.92it/s]

Mapping 'aspects_of_the_public_transport_expected': 100%|██████████| 1527/1527 [00:00<00:00, 39163.39it/s]

Mapping 'aspects_of_the_public_transport_judgment': 100%|██████████| 1527/1527 [00:00<00:00, 39152.85it/s]
Mapping 'available_m


Done ✅
